# Immich Smart Tagging Platform — DevOps Deployment Notebook

**Run every cell in order, top to bottom.**  
Each section has a short explanation of what it does and why.

| Section | What it does |
|---------|-------------|
| 0 | One-time config — fill in YOUR values here |
| 1 | Install Terraform + Ansible |
| 2 | Create Chameleon lease + provision VMs with Terraform |
| 3 | Configure Ansible, verify connectivity |
| 4 | Pre-K8s setup (firewall, Docker, swap) |
| 5 | Install Kubernetes with Kubespray (~30-60 min) |
| 6 | Deploy all services (Immich + MLflow + MinIO) |
| 7 | Verify everything is running |
| 8 | Tear down when done |

> **Repo:** https://github.com/CRUZ773/MLOPS-Immich-Smart-Tagging-Platform.git


---
## Section 0: Your Configuration

**Fill in the variables in the cell below before running anything else.**  
These are the only values you need to change throughout the whole notebook.


In [ ]:
# ─────────────────────────────────────────────────────────────
# FILL IN YOUR VALUES HERE — edit this cell before running anything
# ─────────────────────────────────────────────────────────────

YOUR_NET_ID       = "YOURNETID"           # e.g. "jc1234"
YOUR_KEY_NAME     = "id_rsa_chameleon"    # name of your SSH key in Chameleon
YOUR_PROJECT_NAME = "CHI-XXXXXX"         # your Chameleon project, e.g. CHI-231095
YOUR_SSH_KEY_PATH = "~/.ssh/id_rsa_chameleon"  # path to private key on THIS machine

# Repo URL (already set — don't change)
REPO_URL = "https://github.com/CRUZ773/MLOPS-Immich-Smart-Tagging-Platform.git"
REPO_DIR = "/work/immich-iac"

print("✓ Config loaded")
print(f"  Net ID:      {YOUR_NET_ID}")
print(f"  Key name:    {YOUR_KEY_NAME}")
print(f"  Project:     {YOUR_PROJECT_NAME}")
print(f"  Repo dir:    {REPO_DIR}")


---
## Section 1: Install Terraform and Ansible

We install both tools into `/work/.local` so they persist across sessions in the Chameleon Jupyter environment.


In [ ]:
import os, subprocess

# Add local bin to PATH for this session
os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"

print("Installing Terraform...")
os.makedirs("/work/.local/bin", exist_ok=True)
subprocess.run([
    "bash", "-c",
    "cd /tmp && "
    "wget -q https://releases.hashicorp.com/terraform/1.14.4/terraform_1.14.4_linux_amd64.zip && "
    "unzip -o -q terraform_1.14.4_linux_amd64.zip && "
    "mv terraform /work/.local/bin/ && "
    "rm terraform_1.14.4_linux_amd64.zip"
], check=True)
print("✓ Terraform installed")


In [ ]:
import subprocess, os
os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"

print("Installing Ansible (this takes ~2 min)...")
subprocess.run([
    "pip", "install", "--user", "--quiet",
    "ansible-core==2.16.9", "ansible==9.8.0"
], env={**os.environ, "PYTHONUSERBASE": "/work/.local"}, check=True)
print("✓ Ansible installed")


In [ ]:
# Verify both tools are available
import subprocess, os
os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"

tf = subprocess.run(["terraform", "version"], capture_output=True, text=True)
ans = subprocess.run(["ansible", "--version"], capture_output=True, text=True)

print("Terraform:", tf.stdout.split("\n")[0])
print("Ansible:  ", ans.stdout.split("\n")[0])
print()
print("✓ Both tools ready")


---
## Section 2: Clone Repo and Provision VMs with Terraform

First we clone the IaC repo (which has all our Terraform + Ansible + K8s files).  
Then we create a Chameleon lease for 3 VMs and use Terraform to provision them.

> **Before running the Terraform cell:** you need to edit `clouds.yaml` with your real Chameleon credentials.  
> Instructions are in the cell below.


In [ ]:
import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"

# Clone with --recurse-submodules to pull Kubespray automatically
print(f"Cloning repo to {REPO_DIR} ...")
result = subprocess.run(
    ["git", "clone", "--recurse-submodules", REPO_URL, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Repo cloned successfully")
elif "already exists" in result.stderr:
    print("✓ Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "submodule", "update", "--init", "--recursive"], check=True)
else:
    print("STDERR:", result.stderr)
    raise RuntimeError("Clone failed")


### STOP — Fill in clouds.yaml before continuing

1. Go to **KVM@TACC Horizon GUI** → Identity → Application Credentials → **Create Application Credential**
   - Name: `immich-lab`
   - Expiration: your project due date (UTC time)
   - Click Create → **Download clouds.yaml**

2. Open the downloaded file — it contains your `application_credential_id` and `application_credential_secret`

3. Run the cell below — it will open a text editor where you paste in your credentials


In [ ]:
# This writes the clouds.yaml template to the right location.
# EDIT the file after running this cell — replace the PLACEHOLDER values
# with your real credential_id and credential_secret from the downloaded file.

clouds_yaml_path = f"{REPO_DIR}/tf/kvm/clouds.yaml"

template = """clouds:
  openstack:
    auth:
      auth_url: https://kvm.tacc.chameleoncloud.org:5000
      application_credential_id: "REPLACE_WITH_YOUR_CREDENTIAL_ID"
      application_credential_secret: "REPLACE_WITH_YOUR_CREDENTIAL_SECRET"
    region_name: "KVM@TACC"
    interface: "public"
    identity_api_version: 3
    auth_type: "v3applicationcredential"
"""

with open(clouds_yaml_path, "w") as f:
    f.write(template)

print(f"✓ Template written to {clouds_yaml_path}")
print()
print("NOW: Open the Jupyter file browser on the left side.")
print(f"Navigate to: immich-iac/tf/kvm/clouds.yaml")
print("Replace REPLACE_WITH_YOUR_CREDENTIAL_ID and REPLACE_WITH_YOUR_CREDENTIAL_SECRET")
print("with the values from the clouds.yaml you downloaded from Chameleon.")
print()
print("Then come back and run the next cell.")


In [ ]:
# Verify clouds.yaml has been filled in (not still a placeholder)
with open(f"{REPO_DIR}/tf/kvm/clouds.yaml") as f:
    content = f.read()

if "REPLACE_WITH_YOUR_CREDENTIAL_ID" in content:
    print("❌ clouds.yaml still has placeholder values — edit it first!")
else:
    print("✓ clouds.yaml looks filled in — continuing")


### Create a Chameleon lease for 3 VMs

A "lease" reserves compute resources for you on Chameleon. We need 3x m1.large VMs.


In [ ]:
import subprocess, os

os.environ["OS_AUTH_URL"]    = "https://kvm.tacc.chameleoncloud.org:5000/v3"
os.environ["OS_PROJECT_NAME"]= YOUR_PROJECT_NAME
os.environ["OS_REGION_NAME"] = "KVM@TACC"

lease_name = f"lease_immich_{YOUR_NET_ID}"

# Get m1.large flavor ID
result = subprocess.run(
    ["openstack", "flavor", "show", "m1.large", "-f", "value", "-c", "id"],
    capture_output=True, text=True
)
flavor_id = result.stdout.strip()
print(f"m1.large flavor ID: {flavor_id}")

# Create the lease
import datetime, subprocess
start = (datetime.datetime.utcnow() + datetime.timedelta(seconds=30)).strftime("%Y-%m-%d %H:%M")
end   = (datetime.datetime.utcnow() + datetime.timedelta(hours=12)).strftime("%Y-%m-%d %H:%M")

result = subprocess.run([
    "openstack", "reservation", "lease", "create", lease_name,
    "--start-date", start,
    "--end-date",   end,
    "--reservation", f"resource_type=flavor:instance,flavor_id={flavor_id},amount=3"
], capture_output=True, text=True)

if result.returncode == 0:
    print(f"✓ Lease '{lease_name}' created")
    print(f"  Start: {start} UTC")
    print(f"  End:   {end} UTC")
else:
    print("STDERR:", result.stderr)
    # Lease might already exist
    if "already exists" in result.stderr:
        print("ℹ️  Lease already exists — continuing")
    else:
        raise RuntimeError("Lease creation failed")


In [ ]:
import subprocess, json, os

os.environ["OS_AUTH_URL"]    = "https://kvm.tacc.chameleoncloud.org:5000/v3"
os.environ["OS_PROJECT_NAME"]= YOUR_PROJECT_NAME
os.environ["OS_REGION_NAME"] = "KVM@TACC"

lease_name = f"lease_immich_{YOUR_NET_ID}"

result = subprocess.run(
    ["openstack", "reservation", "lease", "show", lease_name, "-f", "json", "-c", "reservations"],
    capture_output=True, text=True
)
data = json.loads(result.stdout)
RESERVATION_UUID = data["reservations"][0]["flavor_id"]
print(f"✓ Reservation UUID: {RESERVATION_UUID}")
print()
print("This is your TF_VAR_reservation value — saved to RESERVATION_UUID variable.")


### Run Terraform to create the 3 VMs

Terraform reads our configuration in `tf/kvm/` and provisions:
- 3x m1.large VMs on KVM@TACC
- A private network connecting them
- A floating IP on node1 (your public entry point)
- A 50GB block storage volume for MLflow artifacts


In [ ]:
import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"
os.environ["TF_VAR_suffix"]      = YOUR_NET_ID
os.environ["TF_VAR_key"]         = YOUR_KEY_NAME
os.environ["TF_VAR_reservation"] = RESERVATION_UUID

# Unset any OS_ vars that would conflict with Terraform's clouds.yaml auth
for key in list(os.environ.keys()):
    if key.startswith("OS_"):
        del os.environ[key]

tf_dir = f"{REPO_DIR}/tf/kvm"
os.chdir(tf_dir)

print("Running terraform init...")
r = subprocess.run(["terraform", "init", "-input=false"],
                   capture_output=True, text=True, cwd=tf_dir)
print(r.stdout[-500:] if len(r.stdout) > 500 else r.stdout)

print("\nRunning terraform validate...")
r = subprocess.run(["terraform", "validate"],
                   capture_output=True, text=True, cwd=tf_dir)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("Terraform validate failed")
print("✓ Config is valid")


In [ ]:
import subprocess, os, re

# Apply — this actually creates the VMs (takes ~3-5 min)
print("Running terraform apply (creating 3 VMs — takes 3-5 min)...")
print("=" * 60)

tf_dir = f"{REPO_DIR}/tf/kvm"
result = subprocess.run(
    ["terraform", "apply", "-auto-approve", "-input=false"],
    capture_output=True, text=True, cwd=tf_dir
)

if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Terraform apply failed")

print(result.stdout[-1500:])
print("=" * 60)

# Extract floating IP from output
match = re.search(r'node1_floating_ip\s*=\s*"([\d.]+)"', result.stdout)
if match:
    FLOATING_IP = match.group(1)
    print(f"\n⭐ node1 floating IP: {FLOATING_IP}")
    print("Write this down — you'll use it everywhere.")
else:
    # Try terraform output
    r2 = subprocess.run(["terraform", "output", "-raw", "node1_floating_ip"],
                        capture_output=True, text=True, cwd=tf_dir)
    FLOATING_IP = r2.stdout.strip()
    print(f"\n⭐ node1 floating IP: {FLOATING_IP}")


---
## Section 3: Configure Ansible and Verify Connectivity

Ansible connects to your VMs over SSH (jumping through node1) to configure them.  
We need to tell it the floating IP of node1.


In [ ]:
import os

# Write ansible.cfg with the real floating IP substituted in
ansible_cfg = f"""[defaults]
stdout_callback = yaml
inventory = {REPO_DIR}/ansible/inventory.yml

[ssh_connection]
ssh_args = -o ControlMaster=auto -o ControlPersist=60s \
           -o StrictHostKeyChecking=off -o UserKnownHostsFile=/dev/null \
           -o ForwardAgent=yes \
           -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p cc@{FLOATING_IP}"
pipelining = True
"""

cfg_path = f"{REPO_DIR}/ansible/ansible.cfg"
with open(cfg_path, "w") as f:
    f.write(ansible_cfg)

print(f"✓ ansible.cfg written with floating IP: {FLOATING_IP}")
print(f"  Saved to: {cfg_path}")


In [ ]:
import subprocess, os, time

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"
os.environ["ANSIBLE_CONFIG"] = f"{REPO_DIR}/ansible/ansible.cfg"

print("Waiting 30s for VMs to finish booting...")
time.sleep(30)

print("\nTesting connectivity to all 3 nodes...")
result = subprocess.run(
    ["ansible", "-i", f"{REPO_DIR}/ansible/inventory.yml", "all", "-m", "ping"],
    capture_output=True, text=True,
    env={**os.environ, "ANSIBLE_CONFIG": f"{REPO_DIR}/ansible/ansible.cfg"}
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print("\nIf this fails, wait another minute and re-run this cell.")
else:
    print("✓ All nodes reachable")


---
## Section 4: Pre-Kubernetes Configuration

This playbook runs on all 3 nodes and:
- Disables the host firewall (the cloud security groups handle this instead)
- Installs Docker
- Disables swap (required for Kubernetes to work properly)

Takes ~2-3 minutes.


In [ ]:
import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"
os.environ["ANSIBLE_CONFIG"] = f"{REPO_DIR}/ansible/ansible.cfg"

print("Running pre-K8s configuration playbook...")
result = subprocess.run(
    ["ansible-playbook", "-i", f"{REPO_DIR}/ansible/inventory.yml",
     f"{REPO_DIR}/ansible/pre_k8s/pre_k8s_configure.yml"],
    text=True,
    env=os.environ
)
if result.returncode != 0:
    raise RuntimeError("Pre-K8s playbook failed — check output above")
print("\n✓ Pre-K8s configuration complete")


---
## Section 5: Install Kubernetes with Kubespray

⏱ **This takes 30–60 minutes. Start it and come back.**

Kubespray is an Ansible-based tool that installs a production-grade Kubernetes cluster across your 3 nodes.  
It handles everything: container runtime, networking (Calico), DNS (CoreDNS), dashboard, etc.

When it finishes, the PLAY RECAP must show: `failed=0`  
If it shows failures, just re-run this cell — Kubespray is safe to re-run (idempotent).


In [ ]:
import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"
os.environ["ANSIBLE_CONFIG"] = f"{REPO_DIR}/ansible/ansible.cfg"
os.environ["ANSIBLE_ROLES_PATH"] = "roles"

kubespray_dir = f"{REPO_DIR}/ansible/k8s/kubespray"
inventory_dir = f"{REPO_DIR}/ansible/k8s/inventory/mycluster"

print("Starting Kubespray Kubernetes installation...")
print("This will take 30-60 minutes. Watch for the PLAY RECAP at the end.")
print("=" * 60)

result = subprocess.run(
    ["ansible-playbook",
     "-i", inventory_dir,
     "--become", "--become-user=root",
     "./cluster.yml"],
    cwd=kubespray_dir,
    text=True,
    env=os.environ
)

print("=" * 60)
if result.returncode != 0:
    print("\n❌ Kubespray reported failures.")
    print("Check the output above for the failing task.")
    print("Re-run this cell to retry — it is safe to re-run.")
else:
    print("\n✓ Kubernetes installation complete!")
    print("PLAY RECAP showed failed=0")


---
## Section 6: Deploy All Services

This Ansible playbook does everything remaining automatically:
1. Configures `kubectl` for the `cc` user on node1
2. Auto-generates secrets for all services (never stored in Git)
3. Deploys MinIO (object storage for MLflow artifacts)
4. Deploys MLflow (model registry + experiment tracker)
5. Deploys PostgreSQL, Redis, and all Immich containers

At the end it prints the URL for every service.


In [ ]:
import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["PYTHONUSERBASE"] = "/work/.local"
os.environ["ANSIBLE_CONFIG"] = f"{REPO_DIR}/ansible/ansible.cfg"

print("Deploying all services (Immich + MLflow + MinIO)...")
print("Takes ~5 minutes.")
print("=" * 60)

result = subprocess.run(
    ["ansible-playbook", "-i", f"{REPO_DIR}/ansible/inventory.yml",
     f"{REPO_DIR}/ansible/post_k8s/post_k8s_configure.yml"],
    text=True,
    env=os.environ
)

print("=" * 60)
if result.returncode != 0:
    raise RuntimeError("Post-K8s playbook failed — check output above")

print("\n✓ All services deployed!")
print()
print("=" * 50)
print(f"  Immich:  http://{FLOATING_IP}:2283")
print(f"  MLflow:  http://{FLOATING_IP}:8000")
print(f"  MinIO:   http://{FLOATING_IP}:9001")
print("=" * 50)
print()
print("Open these in your browser to confirm they work.")


---
## Section 7: Verify Everything is Running

Run these cells to confirm all pods are healthy before recording your videos.


In [ ]:
import subprocess, os

# SSH into node1 and run kubectl commands
def kubectl(cmd, namespace=None):
    ns = ["-n", namespace] if namespace else ["-A"]
    full_cmd = (
        f"ssh -o StrictHostKeyChecking=no -i {YOUR_SSH_KEY_PATH} "
        f"cc@{FLOATING_IP} 'kubectl get {cmd} {' '.join(ns)}'"
    )
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    return result.stdout + result.stderr

print("=== All pods across the cluster ===")
print(kubectl("pods"))


In [ ]:
print("=== Immich namespace pods ===")
print(kubectl("pods", "immich"))
print()
print("=== Platform namespace pods ===")
print(kubectl("pods", "immich-platform"))


In [ ]:
print("=== Services (ports + external IPs) ===")
print(kubectl("svc", "immich"))
print()
print(kubectl("svc", "immich-platform"))


In [ ]:
print("=== Persistent Volume Claims (proves storage is bound) ===")
print(kubectl("pvc", "immich"))
print()
print(kubectl("pvc", "immich-platform"))
print()
print("STATUS should be 'Bound' for all PVCs.")
print("If any show 'Pending', wait 1-2 min and re-run.")


In [ ]:
# Final summary
import urllib.request

print("Checking service reachability from notebook...")
print()

services = {
    "Immich":  f"http://{FLOATING_IP}:2283",
    "MLflow":  f"http://{FLOATING_IP}:8000",
    "MinIO":   f"http://{FLOATING_IP}:9001",
}

for name, url in services.items():
    try:
        r = urllib.request.urlopen(url, timeout=5)
        print(f"  ✓ {name:8s} → {url}  (HTTP {r.status})")
    except Exception as e:
        print(f"  ⚠ {name:8s} → {url}  ({e})")

print()
print("If all show ✓ — you're ready to record your videos!")
print()
print("VIDEO CHECKLIST:")
print("  Video 1 (Q2.3): kubectl get pods -n immich → browser → http://{FLOATING_IP}:2283")
print("  Video 2 (Q2.4): kubectl get pvc → delete pod → browser → MLflow + MinIO")


---
## Section 8: Tear Down

**Only run this AFTER you have finished recording both videos.**

This destroys all Chameleon resources (VMs, network, block storage) so you don't waste your allocation.


In [ ]:
# ⚠️ ONLY RUN THIS AFTER RECORDING BOTH VIDEOS
# Uncomment the lines below to actually run teardown

import subprocess, os

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]
os.environ["TF_VAR_suffix"]      = YOUR_NET_ID
os.environ["TF_VAR_key"]         = YOUR_KEY_NAME
os.environ["TF_VAR_reservation"] = RESERVATION_UUID

# Unset OS_ vars
for key in list(os.environ.keys()):
    if key.startswith("OS_"):
        del os.environ[key]

tf_dir = f"{REPO_DIR}/tf/kvm"

# UNCOMMENT TO RUN:
# result = subprocess.run(
#     ["terraform", "destroy", "-auto-approve", "-input=false"],
#     cwd=tf_dir, text=True, env=os.environ
# )
# print(result.stdout[-1000:])
# if result.returncode == 0:
#     print("✓ All resources destroyed")
# else:
#     print("Error:", result.stderr)

print("Teardown cell is commented out for safety.")
print("Uncomment the lines above when you are ready to destroy.")
